### Lab: First API Call (Anthropic)
- Step 1: Set `ANTHROPIC_API_KEY` as an environment variable (do not hardcode keys).
- Step 2: Install the SDK if needed.
- Step 3: Run the call, inspect the response, and log token usage and cost.

In [ ]:
# If needed, install the SDK in this environment:
# %pip install anthropic

In [23]:
import os
from anthropic import Anthropic

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("Set ANTHROPIC_API_KEY in your environment.")

client = Anthropic(api_key=api_key)

model = "claude-sonnet-4-6"
prompt = "Summarize the key credit risks in a high-yield corporate bond portfolio in 3 bullet points."

# Accurate token count BEFORE the call — avoids surprise context overflows
token_count = client.messages.count_tokens(
    model=model,
    messages=[{"role": "user", "content": prompt}],
)
print(f"Pre-call token count: {token_count.input_tokens} input tokens")

response = client.messages.create(
    model=model,
    max_tokens=300,
    temperature=0.2,
    messages=[{"role": "user", "content": prompt}],
)

text = response.content[0].text
print("\nResponse:")
print(text)

usage = response.usage
print(f"\nActual usage — Input: {usage.input_tokens}, Output: {usage.output_tokens}")

# claude-sonnet-4-6 pricing: $3.00 input / $15.00 output per 1M tokens
INPUT_PRICE_PER_1M = 3.00
OUTPUT_PRICE_PER_1M = 15.00
cost = (usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_1M + \
       (usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
print(f"Estimated cost (USD): ${cost:.6f}")

Pre-call token count: 29 input tokens

Response:
# Key Credit Risks in a High-Yield Corporate Bond Portfolio

• **Default Risk** — High-yield ("junk") issuers carry sub-investment-grade ratings, meaning they have a materially higher probability of missing interest or principal payments, particularly during economic downturns or periods of financial stress when refinancing becomes difficult.

• **Liquidity & Market Risk** — High-yield bonds trade in thinner, less liquid markets than investment-grade debt, making it harder to exit positions at fair value during volatility; spread widening can cause significant mark-to-market losses even without an actual default.

• **Credit Deterioration & Refinancing Risk** — Issuers may face ratings downgrades driven by weakening fundamentals, rising leverage, or sector headwinds; many high-yield companies rely on periodic debt rollovers, and a tightening credit environment can make refinancing costly or impossible, accelerating distress.

Actual usag

### Lab: First API Call (OpenAI)
- Step 1: Set `OPENAI_API_KEY` as an environment variable.
- Step 2: Install the SDK if needed: `pip install openai`
- Step 3: Run the call, inspect the response, and log token usage and cost.
- Note: OpenAI uses `choices[0].message.content` and `usage.prompt_tokens` / `usage.completion_tokens` — different field names from Anthropic.

In [ ]:
import os

# Requires: pip install openai
# Set OPENAI_API_KEY as an environment variable (never hardcode keys)

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    print("OPENAI_API_KEY not set — skipping. Set it to run this cell.")
else:
    from openai import OpenAI

    openai_client = OpenAI(api_key=openai_api_key)

    openai_model = "gpt-4o-mini"
    openai_prompt = "Summarize the key risks in a high-yield corporate bond portfolio in 3 bullet points."

    openai_response = openai_client.chat.completions.create(
        model=openai_model,
        max_tokens=300,
        temperature=0.2,
        messages=[{"role": "user", "content": openai_prompt}],
    )

    openai_text = openai_response.choices[0].message.content
    print("Response:")
    print(openai_text)

    usage = openai_response.usage
    print(f"\nActual usage — Input: {usage.prompt_tokens}, Output: {usage.completion_tokens}")

    # gpt-4o-mini pricing: $0.15 input / $0.60 output per 1M tokens
    INPUT_PRICE_PER_1M = 0.15
    OUTPUT_PRICE_PER_1M = 0.60
    cost = (usage.prompt_tokens / 1_000_000) * INPUT_PRICE_PER_1M + \
           (usage.completion_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    print(f"Estimated cost (USD): ${cost:.6f}")

#### Rate Limits and Retry Handling
- Providers enforce RPM (requests per minute) and TPM (tokens per minute) quotas.
- Exceeding them returns HTTP 429 — your code must handle this gracefully.
- Pattern: exponential backoff with jitter — each retry waits 2× longer plus a small random offset.
- Finance use case: nightly batch jobs processing thousands of invoices or trade summaries must be rate-limit-aware or they will silently drop records.

In [25]:
import time
import random

def call_with_retry(client, model, messages, max_retries=5, base_delay=1.0):
    """
    Anthropic API call with exponential backoff on HTTP 429 (rate limit).
    Use this for any batch job: invoice processing, patient record summaries,
    student report generation, or bulk marketing content creation.
    """
    for attempt in range(max_retries):
        try:
            return client.messages.create(
                model=model,
                max_tokens=200,
                messages=messages,
            )
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                if attempt == max_retries - 1:
                    raise
                delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                print(f"Rate limit hit. Retrying in {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                raise

# Multi-domain batch examples that would hit rate limits at scale
batch_examples = {
    "Finance":    "Summarize risk factors in this loan application: Credit 680, Income $65k, Debt $18k",
    "Healthcare": "Extract diagnosis codes and medications from: 'Patient presents with T2DM, prescribed Metformin 500mg'",
    "Education":  "Generate a personalized study plan for a student scoring 62% in Algebra and 78% in English",
    "Marketing":  "Write a 2-sentence ad copy for a loyalty rewards program targeting millennial customers",
    "HR":         "Score this resume for a data analyst role on communication, SQL, and Python skills",
}

print("Batch jobs per domain — each would hit rate limits at scale:\n")
for domain, task in batch_examples.items():
    print(f"[{domain}] {task[:80]}...")

print("\nUncomment the loop below to process live with retry logic:")
print("# for domain, task in batch_examples.items():")
print("#     resp = call_with_retry(client, 'claude-sonnet-4-6', [{'role':'user','content':task}])")
print("#     print(f'{domain}: {resp.content[0].text[:100]}')")
print("\nRate limit pattern: 1s → 2s → 4s → 8s → 16s (+ jitter to avoid thundering herd)")

Batch jobs per domain — each would hit rate limits at scale:

[Finance] Summarize risk factors in this loan application: Credit 680, Income $65k, Debt $...
[Healthcare] Extract diagnosis codes and medications from: 'Patient presents with T2DM, presc...
[Education] Generate a personalized study plan for a student scoring 62% in Algebra and 78% ...
[Marketing] Write a 2-sentence ad copy for a loyalty rewards program targeting millennial cu...
[HR] Score this resume for a data analyst role on communication, SQL, and Python skil...

Uncomment the loop below to process live with retry logic:
# for domain, task in batch_examples.items():
#     resp = call_with_retry(client, 'claude-sonnet-4-6', [{'role':'user','content':task}])
#     print(f'{domain}: {resp.content[0].text[:100]}')

Rate limit pattern: 1s → 2s → 4s → 8s → 16s (+ jitter to avoid thundering herd)
